# 14 — Silver Validation

**Objetivo:** ejecutar validaciones cruzadas sobre todos los artefactos Silver construidos en 10–13: artifacts, lineage, schema version, grains, FK coverage y consistencia de entity resolution.

In [ ]:
import pandas as pd
from argos.common.paths import ProjectPaths
from argos.config.silver import load_silver_spec
from argos.silver.validation import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover(); paths.ensure_runtime_dirs()
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

for report_name in ["10_validation.parquet", "11_validation.parquet", "12_validation.parquet", "13_validation.parquet"]:
    frame = pd.read_parquet(paths.reports / "silver" / report_name)
    blocking = frame.loc[frame["blocking"]]
    assert not blocking.empty and blocking["passed"].all(), f"Prerequisite failed: {report_name}"

{"implementation": BUILD_IMPLEMENTATION, "schema_version": SILVER_SPEC.schema_version}

In [ ]:
from argos.silver.validation import validate_silver_m2

result = validate_silver_m2(
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    project_config_path=paths.configs / "project.toml",
    silver_spec=SILVER_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
)
result.validation

## Issue register

`m2_issue_register.parquet` contiene únicamente checks que no pasaron. Un warning puede sobrevivir M2 si está explícitamente documentado; un issue `blocking` no.

In [ ]:
result.issues

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("14 ✅ PASS" if result.ready_for_gate else "14 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_gate, "Silver cross-validation remains OPEN; resolve blocking checks before M2 gate."

**Siguiente:** `15_m2_gate.ipynb`.